In [33]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [34]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [35]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [36]:
multiply.name

'multiply'

In [37]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [38]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [70]:
llm = ChatOllama( model = 'qwen3:4b')

In [40]:
llm.invoke('hi')

AIMessage(content='Hello! 😊 How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-06-25T09:39:01.9728743Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11038118200, 'load_duration': 556927400, 'prompt_eval_count': 11, 'prompt_eval_duration': 186125000, 'eval_count': 199, 'eval_duration': 10247096000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019efe25-7fb4-7131-9d9d-bfce4c403834-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 199, 'total_tokens': 210})

In [41]:
llm_with_tools = llm.bind_tools([multiply])

In [42]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="The user is greeting me, but there are no applicable functions for this query. I'll respond directly without using any tools.\n\nHi! I'm doing well, thank you! How can I assist you today?", additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-06-25T09:39:09.9255298Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7910781400, 'load_duration': 509376300, 'prompt_eval_count': 146, 'prompt_eval_duration': 172952000, 'eval_count': 123, 'eval_duration': 7166255000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019efe25-aafc-7053-ae0c-64c2f88178f6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 146, 'output_tokens': 123, 'total_tokens': 269})

In [43]:
llm_with_tools.invoke('hi can u multiply 3 with 5').tool_calls[0]


{'name': 'multiply',
 'args': {'a': 3, 'b': 5},
 'id': 'e6c5ce65-4b01-44ec-8a90-fdad2422e3cb',
 'type': 'tool_call'}

In [44]:
query = HumanMessage('can you multiply 3 with 1000')

In [45]:
messages = [query]

In [46]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [47]:
result = llm_with_tools.invoke(messages)

In [48]:
messages.append(result)

In [49]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-06-25T09:39:30.1990213Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13206408600, 'load_duration': 539290200, 'prompt_eval_count': 153, 'prompt_eval_duration': 282111000, 'eval_count': 212, 'eval_duration': 12371178000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019efe25-e57e-7893-93ff-14b82d08a859-0', tool_calls=[{'name': 'multiply', 'args': {'b': 1000, 'a': 3}, 'id': '0b73b833-7419-4569-aa1c-70cb0bd21167', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 212, 'total_tokens': 365})]

In [50]:
tool_result = multiply.invoke(result.tool_calls[0])

In [51]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='0b73b833-7419-4569-aa1c-70cb0bd21167')

In [52]:
messages.append(tool_result)

In [53]:
llm_with_tools.invoke(messages).content

'3 multiplied by 1000 equals 3000.'

In [54]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/a4e0d7892493e4fd4d3740bb/pair/{base_currency}/{target_currency}'
  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [55]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [56]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782345601,
 'time_last_update_utc': 'Thu, 25 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1782432001,
 'time_next_update_utc': 'Fri, 26 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.6529}

In [57]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [58]:
llm = ChatOllama( model = 'qwen3:4b')

In [59]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [73]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr')]

In [74]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [75]:
ai_message = llm_with_tools.invoke(messages)

In [76]:
messages.append(ai_message)

In [77]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [78]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-06-25T11:10:23.3569782Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3415368618900, 'load_duration': 615635200, 'prompt_eval_count': 232, 'prompt_eval_duration': 134730000, 'eval_count': 2852, 'eval_duration': 3414745741000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019efe45-3130-7d73-8520-ac53a79e4faa-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '8cbf3ab1-ccf1-47e7-83ae-de596aae83c2', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 94.6529}, 'id': '40736fc1-5d5d-4b67-bb9f-119784089870', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

In [79]:
llm_with_tools.invoke(messages).content

'The conversion factor from USD to INR is **94.6529** (1 USD = 94.6529 INR). \n\nUsing this rate, **10 USD** converts to **946.529 INR**. \n\nThis result is derived from the provided exchange rate data and calculations.'